# **Neural Networks Day 5 - PyTorch RNN vs Feedforward (Google Stock Price prediction)**

Welcome to this Neural Networks Day 5 teaching notebook!

In this notebook we will:
- Load and explore Google stock price development
- Visualize temporal evolution and data distribution
- Build a **Feedforward baseline** in PyTorch
- Build a **Simple RNN** in PyTorch
- Train and evaluate both models for next-step stock price prediction


## **Module Imports**

We keep the stack simple for teaching purposes:
- `csv`, `datetime`, `numpy`: data loading and preprocessing
- `matplotlib`: visualization
- `torch`: model definition, training, and evaluation


In [ ]:
import csv
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## **Load the Google Stock Dataset**

We parse the CSV directly and keep the core variables:
- `dates`
- `open_prices`
- `high`, `low`, `close`, `volume`


In [ ]:
# needs to be in ./data/
csv_path = "./data/Google_Stock_Price_Train.csv"

dates = []
open_prices = []
high_prices = []
low_prices = []
close_prices = []
volumes = []

with open(csv_path, "r", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        dates.append(datetime.strptime(row["Date"], "%m/%d/%Y"))
        open_prices.append(float(row["Open"]))
        high_prices.append(float(row["High"]))
        low_prices.append(float(row["Low"]))
        close_prices.append(float(row["Close"].replace(",", "")))
        volumes.append(int(row["Volume"].replace(",", "")))

open_prices = np.array(open_prices, dtype=np.float32)
high_prices = np.array(high_prices, dtype=np.float32)
low_prices = np.array(low_prices, dtype=np.float32)
close_prices = np.array(close_prices, dtype=np.float32)
volumes = np.array(volumes, dtype=np.float32)

print(f"Rows: {len(open_prices)}")
print(f"Date range: {dates[0].date()} -> {dates[-1].date()}")
print(f"Open min/max: {open_prices.min():.2f}/{open_prices.max():.2f}")

## **Visualize Temporal Evolution and Data Distribution**

First, inspect the evolution over time. Then inspect distributions to understand scale and skew.


In [ ]:
returns = np.diff(open_prices) / open_prices[:-1]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].plot(dates, open_prices, color="tab:blue", linewidth=1.2)
axes[0, 0].set_title("Open Price Over Time")
axes[0, 0].set_xlabel("Date")
axes[0, 0].set_ylabel("Price")

axes[0, 1].hist(open_prices, bins=40, color="tab:green", alpha=0.85)
axes[0, 1].set_title("Distribution of Open Prices")
axes[0, 1].set_xlabel("Open Price")
axes[0, 1].set_ylabel("Count")

axes[1, 0].hist(returns, bins=40, color="tab:orange", alpha=0.85)
axes[1, 0].set_title("Distribution of Daily Returns (Open)")
axes[1, 0].set_xlabel("Return")
axes[1, 0].set_ylabel("Count")

axes[1, 1].plot(dates, volumes, color="tab:red", linewidth=1.0)
axes[1, 1].set_title("Trading Volume Over Time")
axes[1, 1].set_xlabel("Date")
axes[1, 1].set_ylabel("Volume")

plt.tight_layout()
plt.show()

## **Prepare Time-Series Supervised Data**

We use a sliding window of `window_size` past days to predict the next day open price.

### Temporal split (no shuffling)
- First 80% for training
- Last 20% for evaluation

To avoid data leakage:
- Fit min-max normalization only on training open prices
- Apply the same scaling to test period


In [ ]:
window_size = 100
split_idx = int(0.8 * len(open_prices))

train_open = open_prices[:split_idx]
test_open = open_prices[split_idx:]

train_min = train_open.min()
train_max = train_open.max()

# Numerical safety for very small ranges
eps = 1e-8

def minmax_scale(x, x_min, x_max):
    return (x - x_min) / (x_max - x_min + eps)

def minmax_inverse(x_scaled, x_min, x_max):
    return x_scaled * (x_max - x_min + eps) + x_min

scaled_all = minmax_scale(open_prices, train_min, train_max)

def make_sequences(series_1d, window):
    X, y = [], []
    for i in range(window, len(series_1d)):
        X.append(series_1d[i-window:i])
        y.append(series_1d[i])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

# Train sequences from train-only period
X_train, y_train = make_sequences(scaled_all[:split_idx], window_size)

# Test sequences need historical context (last window_size points from train)
test_context_start = split_idx - window_size
X_test, y_test = make_sequences(scaled_all[test_context_start:], window_size)

# Shapes for FFNN: (batch, window)
X_train_ff = torch.tensor(X_train)
y_train_t = torch.tensor(y_train).unsqueeze(1)
X_test_ff = torch.tensor(X_test)
y_test_t = torch.tensor(y_test).unsqueeze(1)

# Shapes for RNN: (batch, seq_len, input_size=1)
X_train_rnn = torch.tensor(X_train).unsqueeze(-1)
X_test_rnn = torch.tensor(X_test).unsqueeze(-1)

print("Train samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])
print("FFNN input shape:", X_train_ff.shape)
print("RNN input shape:", X_train_rnn.shape)

## **Model 1: Feedforward Baseline (PyTorch)**

A simple baseline:
- Input: flattened `window_size` values
- Hidden layer + ReLU
- Output: one value (next scaled open price)


In [ ]:
class FeedforwardRegressor(nn.Module):
    def __init__(self, input_dim=60, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        return self.net(x)

## **Model 2: Simple RNN (PyTorch)**

The RNN processes the sequence one step at a time and uses the final hidden state for prediction.


In [ ]:
class SimpleRNNRegressor(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1):
        super().__init__()
		# NOTE: feel free to play around with the definition of the RNN layers! Can also use LSTM or GRU, add more layers, etc.
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            nonlinearity="tanh"
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x shape: (batch, seq_len, input_size)
        out, _ = self.rnn(x)
        last_hidden = out[:, -1, :]  # final time step representation
        return self.fc(last_hidden)

## **Training & Evaluation Utilities**

We use the same training loop for both models.
Metric comparison on unscaled values:
- MSE
- MAE


In [ ]:
def train_model(model, train_loader, epochs=40, lr=1e-3):
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = []
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        history.append(epoch_loss)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:>2}/{epochs} - loss: {epoch_loss:.6f}")
    return history

def evaluate_model(model, X_test_tensor, y_test_tensor, x_min, x_max):
    model.eval()
    with torch.no_grad():
        preds_scaled = model(X_test_tensor.to(device)).cpu().numpy().squeeze()

    y_true_scaled = y_test_tensor.numpy().squeeze()

    preds = minmax_inverse(preds_scaled, x_min, x_max)
    y_true = minmax_inverse(y_true_scaled, x_min, x_max)

    mse = np.mean((preds - y_true) ** 2)
    mae = np.mean(np.abs(preds - y_true))
    return mse, mae, preds, y_true

## **Train Both Models**


In [ ]:
batch_size = 32
epochs = 50
learning_rate = 1e-2

# DataLoaders
tf_train = TensorDataset(X_train_ff, y_train_t)
tr_train = TensorDataset(X_train_rnn, y_train_t)

ff_loader = DataLoader(tf_train, batch_size=batch_size, shuffle=True)
rnn_loader = DataLoader(tr_train, batch_size=batch_size, shuffle=True)

# Instantiate
ff_model = FeedforwardRegressor(input_dim=window_size, hidden_dim=64)
rnn_model = SimpleRNNRegressor(input_size=1, hidden_size=32, num_layers=1)

print("Training Feedforward model")
ff_history = train_model(ff_model, ff_loader, epochs=epochs, lr=learning_rate)

print("Training RNN model")
rnn_history = train_model(rnn_model, rnn_loader, epochs=epochs, lr=learning_rate)


## **Question 5: Evaluate and Compare**


In [ ]:
ff_mse, ff_mae, ff_preds, y_true = evaluate_model(
    ff_model, X_test_ff, y_test_t, train_min, train_max
)
rnn_mse, rnn_mae, rnn_preds, _ = evaluate_model(
    rnn_model, X_test_rnn, y_test_t, train_min, train_max
)

print("Feedforward -> MSE: {:.4f}, MAE: {:.4f}".format(ff_mse, ff_mae))
print("RNN -> MSE: {:.4f}, MAE: {:.4f}".format(rnn_mse, rnn_mae))

In [ ]:
# Learning curves
plt.figure(figsize=(10, 4))
plt.plot(ff_history, label="Feedforward train loss")
plt.plot(rnn_history, label="RNN train loss")
plt.title("Training Loss Curves")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

# Prediction comparison on test period
test_dates = dates[split_idx:]

plt.figure(figsize=(12, 5))
plt.plot(test_dates, y_true, label="True Open", linewidth=2)
plt.plot(test_dates, ff_preds, label="Feedforward Prediction", alpha=0.9)
plt.plot(test_dates, rnn_preds, label="RNN Prediction", alpha=0.9)
plt.title("Google Stock Open Price: True vs Predicted (Test Period)")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## **Question 6: Effect of Window Size**

We now compare FFNN vs RNN across different `window_size` values.

- FFNN can be strong for short windows.
- For longer windows, sequence models (RNN) should become more meaningful.

We include an FFNN setting where hidden width follows an inverse rule:
`hidden_dim = max(8, int(512 / window_size))`.
This makes FFNN capacity shrink as window size grows.


In [ ]:
def prepare_tensors_for_window(window_size, split_ratio=0.8):
    split_idx = int(split_ratio * len(open_prices))

    train_open = open_prices[:split_idx]
    train_min = train_open.min()
    train_max = train_open.max()

    scaled_all = minmax_scale(open_prices, train_min, train_max)

    X_train, y_train = make_sequences(scaled_all[:split_idx], window_size)
    X_test, y_test = make_sequences(scaled_all[split_idx - window_size:], window_size)

    X_train_ff = torch.tensor(X_train)
    X_test_ff = torch.tensor(X_test)
    X_train_rnn = torch.tensor(X_train).unsqueeze(-1)
    X_test_rnn = torch.tensor(X_test).unsqueeze(-1)

    y_train_t = torch.tensor(y_train).unsqueeze(1)
    y_test_t = torch.tensor(y_test).unsqueeze(1)

    return X_train_ff, X_test_ff, X_train_rnn, X_test_rnn, y_train_t, y_test_t, train_min, train_max

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


In [ ]:
def run_window_experiment(window_sizes, ff_rule="inverse", epochs=25, batch_size=32, lr=1e-3):
    results = []

    for w in window_sizes:
        X_train_ff, X_test_ff, X_train_rnn, X_test_rnn, y_train_t, y_test_t, train_min, train_max = prepare_tensors_for_window(w)

        ff_hidden = max(8, int(512 / w)) if ff_rule == "inverse" else 64

        ff_model = FeedforwardRegressor(input_dim=w, hidden_dim=ff_hidden)
        rnn_model = SimpleRNNRegressor(input_size=1, hidden_size=32, num_layers=1)

        ff_loader = DataLoader(TensorDataset(X_train_ff, y_train_t), batch_size=batch_size, shuffle=True)
        rnn_loader = DataLoader(TensorDataset(X_train_rnn, y_train_t), batch_size=batch_size, shuffle=True)

        _ = train_model(ff_model, ff_loader, epochs=epochs, lr=lr)
        _ = train_model(rnn_model, rnn_loader, epochs=epochs, lr=lr)

        ff_mse, ff_mae, _, _ = evaluate_model(ff_model, X_test_ff, y_test_t, train_min, train_max)
        rnn_mse, rnn_mae, _, _ = evaluate_model(rnn_model, X_test_rnn, y_test_t, train_min, train_max)

        results.append({
            "window": w,
            "ff_hidden": ff_hidden,
            "ff_params": count_parameters(ff_model),
            "rnn_params": count_parameters(rnn_model),
            "ff_mae": ff_mae,
            "rnn_mae": rnn_mae,
            "ff_mse": ff_mse,
            "rnn_mse": rnn_mse,
        })

    return results


In [ ]:
window_grid = [5, 50, 100, 200]

# Use inverse FF rule to reflect 1/window_size teaching setup
window_results = run_window_experiment(window_grid, ff_rule="inverse", epochs=25, batch_size=32, lr=1e-3)

print("window | ff_hidden | ff_params | rnn_params | ff_MAE | rnn_MAE")
print("-" * 66)
for r in window_results:
    print(f"{r['window']:>6} | {r['ff_hidden']:>9} | {r['ff_params']:>9} | {r['rnn_params']:>10} | {r['ff_mae']:.4f} | {r['rnn_mae']:.4f}")

w = [r["window"] for r in window_results]
ff_mae = [r["ff_mae"] for r in window_results]
rnn_mae = [r["rnn_mae"] for r in window_results]
ff_params = [r["ff_params"] for r in window_results]
rnn_params = [r["rnn_params"] for r in window_results]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(w, ff_mae, marker='o', label='FFNN MAE')
axes[0].plot(w, rnn_mae, marker='o', label='RNN MAE')
axes[0].set_title('Test MAE vs Window Size')
axes[0].set_xlabel('window_size')
axes[0].set_ylabel('MAE')
axes[0].grid(alpha=0.2)
axes[0].legend()

axes[1].plot(w, ff_params, marker='o', label='FFNN params')
axes[1].plot(w, rnn_params, marker='o', label='RNN params')
axes[1].set_title('Parameter Count vs Window Size')
axes[1].set_xlabel('window_size')
axes[1].set_ylabel('Number of parameters')
axes[1].grid(alpha=0.2)
axes[1].legend()

plt.tight_layout()
plt.show()